In [1]:
import os
from dotenv import load_dotenv
import time

from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

# from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain.agents import create_agent

from langchain_core.tools import Tool
from langchain_core.tools import tool

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langgraph.graph import StateGraph, START, END
from typing import TypedDict


/Users/sunahgwak/Documents/Repository/source/ollama/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/_f/6s2bl8nd4s576jgt_yh7n8bc0000gn/T/ipykernel_31566/1528809645.py:20: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader, DirectoryLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:

load_dotenv()

apiKey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.environ["HF_TOKEN"]
COHERE_API_KEY = os.environ["COHERE_API_KEY"]
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

watson_llm = ChatWatsonx(
  model_id="ibm/granite-4-h-small",
  url=f"{watsonx_ai_url}",
  api_key = f"{apiKey}",
  project_id=f"{project_id}",
  max_tokens = 2000,
  params = {
    "temperature":0
  }
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apiKey}",
    project_id=f"{project_id}"
)

parser = StrOutputParser()


#### LangGraph
- 워크플로우 프레임워크
- LCEL 선형(A -> B -> C) / LangChain 순환 (A->B->A->C->B) 흐름 지원
- 개념
  - Node: 실행할 함수
  - Edge: 노드 간 연결
  - State: 전체 상태를 담는 딕셔너리
- 조건부 엣지, Agent 루프, 자기 수정, 멀티 에이전트 등 복잡한 패턴 구현

In [ ]:
%pip install langgraph grandalf

In [5]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

#### TypedDict / Pydantic
- TypedDict: 딕셔너리 키, 값 타입을 정의할 수 있게 도와줌, 단순 State 구현 시 사용, 기본값 줄 수 없음
- Pydantic: validation검사(입력값), 검증이 필요한 State 인 경우 사용

In [9]:
# 데이터 저장소 생성
# message 상태관리 할거야 -> 공유
class MyState(TypedDict):
  message:str

# 작업 함수 생성(노드) / state무조건 받아야 한다
def say_hello(state):
  # return 대상은 state 값의 변화를 처리
  # message에 Hello, LangGraph가 들어간다
  print(f"초기값: {state}")
  return {"message": "Hello, LangGraph!!!!"}

# 그래프 생성
graph = StateGraph(MyState)
graph.add_node("hello", say_hello) # hello node만들기 

# 그래프 연결
graph.add_edge(START, "hello") # hello로 간다 (say_hello 실행)
graph.add_edge("hello", END) # hello로 끝

# 실행
app = graph.compile()
result = app.invoke({"message":"ddd"})  # 초기값
print(result)

# 그래프 시각화(참고)
app.get_graph().print_ascii()

초기값: {'message': 'ddd'}
{'message': 'Hello, LangGraph!!!!'}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | hello |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


#### State
- 상태를 중심으로 동작
- TypedDict, Pydantic Base Model을 사용하여 정의
- graph가 실행 중 지속적으로 업데이트 됨
- 노드간의 전환은 조건부 엣지를 통해 제어 가능하며 복잡한 의사결정 프로세스 모델링 가능
- 재귀적 실행 지원

#### Node
- 실제 작업을 수행하는 기본 단위
- 함수 기반
- 상태 중심: 현재 상태를 입력으로 받아 처리
- 독립적 힐생: 각 노드는 독립적으로 실행
- 조합 가능: 여러 노드를 연결하여 복잡한 워크플로우 가능

####  Edge
- 노드간의 연결과 실행 흐름을 정의


In [28]:
# 카운터 공유
class CounterState(TypedDict):
  counter:int
  text:str
  
# 증가 함수(노드)
def increment(state):
  # state: 노드 함수에서 상태변수에 접근할 때 
  print(f"state: {state}")
  print(f"현재 카운트: {state['counter']}")
  new_count = state['counter'] + 1
  print(f"새로운 카운트: {new_count}")
  
  # state 값 변경 원하면 return
  return {"counter" : new_count}

# graph 생성
graph = StateGraph(CounterState) 

# node 연결 ("key", 노드)
graph.add_node("increment", increment)

# graph 연결
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

# graph 실행 가능한 형태로 변경
app = graph.compile()
result = app.invoke({"counter":0, "text":"dd"})
print(f"최종 결과 {result}")

# 그래프 시각화
app.get_graph().print_ascii()


state: {'counter': 0, 'text': 'dd'}
현재 카운트: 0
새로운 카운트: 1
최종 결과 {'counter': 1, 'text': 'dd'}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| increment |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [23]:
# 2개의 노드
def first_increment(state):
  counter = state['counter'] 
  new_counter = counter + 1
  print(f"first_increment_counter:{counter}")
  print(f"first_increment_new_counter:{new_counter}")
  return {"counter": new_counter}
  
def second_increment(state):
  counter = state['counter']
  new_counter = counter + 1
  print(f"second_increment_counter:{counter}")
  print(f"second_increment_new_counter:{new_counter}")
  return {"counter":new_counter}
  
# START => first -?> second -> END
graph = StateGraph(CounterState)

graph.add_node("first_increment", first_increment)
graph.add_node("second_increment", second_increment)

graph.add_edge(START, "first_increment")
graph.add_edge("first_increment", "second_increment")
graph.add_edge("second_increment", END)

app = graph.compile()
result = app.invoke({"counter":0})
print(result)

  
app.get_graph().print_ascii()


first_increment_counter:0
first_increment_new_counter:1
second_increment_counter:1
second_increment_new_counter:2
{'counter': 2}
    +-----------+    
    | __start__ |    
    +-----------+    
          *          
          *          
          *          
+-----------------+  
| first_increment |  
+-----------------+  
          *          
          *          
          *          
+------------------+ 
| second_increment | 
+------------------+ 
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


- 조건부 엣지
  - 런타임 상태에 따라 동적으로 실행 경로 결정
  - add_conditional_edges()

In [29]:
# 입력숫자 >10 => big
# 입력숫자 <10 => small

# State: 숫자, 결과
class NumberState(TypedDict):
  number: int
  result: str
  
# node
def handle_big_number(state):
  return {'result':f"{state['number']}는 큰 숫자입니다"}

def handle_small_number(state):
  return {'result':f"{state['number']}는 작은 숫자입니다"}

# router(조건함수)
def check_size(state):
  if state['number'] >  10:
    return "big"
  else:
    return "small"


# 그래프 생성
graph = StateGraph(NumberState)
graph.add_node("big_handler",handle_big_number)
graph.add_node("small_handler",handle_small_number)

# 엣지
graph.add_edge("big_handler", END)
graph.add_edge("small_handler", END)

# 상태에 따라 엣지 변함
# 조건부 엣지(number에 따라 big or small)
# 라우터의 결과에 따라 어디로 갈 것인지 명시
graph.add_conditional_edges(START, check_size, {"big": "big_handler", "small": "small_handler"})


app = graph.compile()
result = app.invoke({"number":15, "result":""})
print(result)

app.get_graph().print_ascii()

{'number': 15, 'result': '15는 큰 숫자입니다'}
              +-----------+                 
              | __start__ |                 
              +-----------+                 
              ..           ..               
            ..               ..             
          ..                   ..           
+-------------+           +---------------+ 
| big_handler |           | small_handler | 
+-------------+           +---------------+ 
              **           **               
                **       **                 
                  **   **                   
                +---------+                 
                | __end__ |                 
                +---------+                 


In [32]:
# 홀,짝

# State: 숫자, 결과
class NumberState(TypedDict):
  number: int
  result: str
  
# node
def even_node(state):
  return {'result':f"{state['number']}는 짝수입니다"}

def odd_node(state):
  return {'result':f"{state['number']}는 홀수입니다"}

# router(조건함수)
def check_number(state):
  if state['number'] % 2 == 0:
    return "even"
  else:
    return "odd"


# 그래프 생성
graph = StateGraph(NumberState)
graph.add_node("even_node",even_node)
graph.add_node("odd_node",odd_node)

# 엣지
graph.add_edge("even_node", END)
graph.add_edge("odd_node", END)

# 조건부 엣지
graph.add_conditional_edges(START, check_number, {"even": "even_node", "odd": "odd_node"})


app = graph.compile()
result = app.invoke({"number":15, "result":""})
print(result)

app.get_graph().print_ascii()

{'number': 15, 'result': '15는 홀수입니다'}
            +-----------+             
            | __start__ |             
            +-----------+             
           ...         ...            
          .               .           
        ..                 ..         
+-----------+           +----------+  
| even_node |           | odd_node |  
+-----------+           +----------+  
           ***         ***            
              *       *               
               **   **                
             +---------+              
             | __end__ |              
             +---------+              


In [37]:
class ScoreState(TypedDict):
  score:int

# score >= 90 : A , >= 80 :B, C
# 노드 return => 상태값 변경
# return을 안하면? None 
def grade_a(state):
  print("A")
  # score = None
  return {} # 안보내거나 비어있는 구조 return
  
def grade_b(state):
  print("B")
  return {}
  
def grade_c(state):
  print("C")
  return {}
  
def route_grade(state):
  if state['score'] >=90:
    return "A"
  elif state['score'] >=80:
    return "B"
  else:
    return "C"
  
graph = StateGraph(ScoreState)
graph.add_node("grade_a", grade_a)
graph.add_node("grade_b", grade_b)
graph.add_node("grade_c", grade_c)

graph.add_edge("grade_a", END)
graph.add_edge("grade_b", END)
graph.add_edge("grade_c", END)
graph.add_conditional_edges(START, route_grade, {"A":"grade_a", "B" :"grade_b", "C": "grade_c"})

app = graph.compile()
result = app.invoke({"score": 95})
print(result)

app.get_graph().print_ascii()

A
{'score': 95}
                     +-----------+                       
                     | __start__ |                       
                   ..+-----------+...                    
               ....         .        ....                
           ....             .            ....            
         ..                 .                ..          
+---------+           +---------+           +---------+  
| grade_a |           | grade_b |           | grade_c |  
+---------+****       +---------+        ***+---------+  
               ****         *        ****                
                   ****     *    ****                    
                       **   *  **                        
                      +---------+                        
                      | __end__ |                        
                      +---------+                        


In [47]:
# 상태관리: text, sentiment, result
# text: 오늘 기분이 너무 좋아
# analyze_sentiment(): 감정평가 text 좋 positive / 싫 negative / x neutral => sentiment 업데이트
# positive_node(): 긍정 의견 / negative_node(): 부정 의견 / neutral_node(): 중립 의견
# route_sentiment: return state['sentiment']

# START => analyze => route_sentiment => positive, negative, neutral


# state
class SentimentState(TypedDict):
  text: str
  sentiment: str
  result: str

# node 
def analyze_sentiment(state):
  text = state['text'] 

  if "좋" in text:
    sentiment = "positive"
  elif "싫" in text:
    sentiment = "negative"
  else:
    sentiment = "neutral"
  
  return {"sentiment": sentiment}

def positive_node(state):
  return {"result":"긍정의견"}

def negative_node(state):
  return {"result":"부정의견"}

def neutral_node(state):
  return {"result":"중립의견"}

# router
def route_sentiment(state):
  return state['sentiment']
  
  
graph = StateGraph(SentimentState)

graph.add_node("analyze_sentiment", analyze_sentiment)
graph.add_node("positive_node",positive_node)
graph.add_node("negative_node",negative_node)
graph.add_node("neutral_node",neutral_node)

graph.add_edge(START, "analyze_sentiment")
graph.add_edge("positive_node", END)
graph.add_edge("negative_node", END)
graph.add_edge("neutral_node", END)

graph.add_conditional_edges("analyze_sentiment", route_sentiment, {"positive":"positive_node", "negative":"negative_node", "neutral":"neutral_node"})

app = graph.compile()
print(app.invoke({"text":"오늘 기분이 너무 좋아"}))
print(app.invoke({"text":"오늘 우울해"}))

app.get_graph().print_ascii()

{'text': '오늘 기분이 너무 좋아', 'sentiment': 'positive', 'result': '긍정의견'}
{'text': '오늘 우울해', 'sentiment': 'neutral', 'result': '중립의견'}
                              +-----------+                              
                              | __start__ |                              
                              +-----------+                              
                                    *                                    
                                    *                                    
                                    *                                    
                          +-------------------+                          
                          | analyze_sentiment |                          
                         .+-------------------+.                         
                    .....           .           .....                    
                ....                .                ....                
             ...                    .                    

In [58]:
# 체인 정의
# assign으로 순차적으로 처리
parser = StrOutputParser()

translate_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력:\n{text}")
]) | watson_llm | parser

summarize_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 3문장으로 요약하세요:\n{text}")
]) | watson_llm | parser

sentiment_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 대답하세요:\n{summary}")
]) | watson_llm | parser

# graph로 만들기

# State 정의
class AnalysisState(TypedDict):
  text: str
  translated: str
  summary:str
  sentiment: str
  done: bool

# node 정의
def translated_node(state):
  result = translate_chain.invoke({"text": state["text"]})
  return {"translated": result}

def summary_node(state):
  result = summarize_chain.invoke({"text":state["translated"]})
  return {"summary": result}

def sentiment_node(state):
  result = sentiment_chain.invoke({"summary": state["summary"]})
  return {"sentiment": result}

graph = StateGraph(AnalysisState)
graph.add_node("translate", translated_node)
graph.add_node("summarize", summary_node)
graph.add_node("sentiment", sentiment_node)

graph.add_edge(START, "translate")
graph.add_edge("translate", "summarize")
graph.add_edge("summarize", "sentiment")
graph.add_edge("sentiment", END)

# 그래프 실행
app = graph.compile()
result = app.invoke({"text":"Python is great!!", "done": False})
print("translated", result['translated'])
print("summary", result['summary'])
print("sentiment", result['sentiment'])

app.get_graph().print_ascii()

translated 파이썬은 정말 멋져요!!
summary 파이썬은 정말 멋진 프로그래밍 언어입니다!
sentiment 긍정
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| translate |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| summarize |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| sentiment |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [65]:
# 조건부엣지: summary >= 100자 이상 good / 100자 미만 poor

class SummaryState(TypedDict):
  text: str
  summary:str
  quality: str # good or poor
  retries: int
  
def summary_node(state):
  result = summarize_chain.invoke({"text":state["text"]})
  return {"summary":result}

# quality
def check_quality_node(state):
  quality = "good" if len(state["summary"]) >= 100 else "poor"
  return {"quality": quality}

def retry_node(state):
  return { "retries":state["retries"]+1}

def route_by_quality(state):
  if state["quality"] == "poor" and state["retries"] <3:
    return "retry"
  return "done"

graph = StateGraph(SummaryState)
graph.add_node("summary_node", summary_node)
graph.add_node("check_quality_node", check_quality_node)
graph.add_node("retry_node", retry_node)

# START -> summary -> check quality -> 조건부 edge 동작
graph.add_edge(START, "summary_node")
graph.add_edge("summary_node", "check_quality_node")
graph.add_conditional_edges("check_quality_node", route_by_quality, {"retry":"retry_node", "done": END})
graph.add_edge("retry_node","summary_node")

app = graph.compile()
result = app.invoke({"text":"짧은 글"})
print(f"최종 요약 ({len(result['summary'])}자): {result['summary'][:100]}")
app.get_graph().print_ascii()

최종 요약 (231자): 네, 다음은 3문장으로 요약한 짧은 글입니다:

1. 짧은 글은 글쓰기의 한 형태로, 명확성, 간결함, 구체성을 강조합니다.
2. 짧은 글은 일반적으로 1,000개 이하의 단어로 
                   +-----------+                 
                   | __start__ |                 
                   +-----------+                 
                          *                      
                          *                      
                          *                      
                  +--------------+               
                  | summary_node |               
                  +--------------+               
                 ***            ***              
               **                  **            
             **                      **          
+--------------------+                 **        
| check_quality_node |                  *        
+--------------------+..                *        
           .            ....            *        
           .                .....       *        
           .                     ..

In [ ]:
# s-c 자기수정

class VerifyState(TypedDict):
  question: str
  answer:str
  feedback: str 
  is_verified: bool
  attempt: int
  
def generate(state):
  """답변을 생성합니다. 이전 피드백이 있으면 반영합니다."""
  prompt = f"질문: {state['question']}"
  
  if state['feedback']:
    prompt += f"\n이전 답변의 피드백: {state['feedback']}\n위 피드백을 반영하여 개선된 답변을 작성하세요"
  
  result = watson_llm.invoke(prompt)
  return {'answer': result.content, "attempt":state['attempt']+1}
  
def verify(state):
  """답변의 정확성과 완전성을 검증합니다."""
  verification =watson_llm.invoke(f"""
                    다음 답변의 정확성과 완전성을 검증하세요
                    
                    질문:
                    {state['question']}
                    
                    답변:
                    {state['answer']}
                    
                    정확하고 완전하면 첫 줄에 'PASS' , 수정이 필요하면 첫줄에 'FAIL'을 쓰고 구체적인 개선 사항을 설명하세요.
                    """)
  
  content = verification.content
  is_pass = content.strip().startswith("PASS")
  return {"is_verified": is_pass, "feedback":content}

# router
def should_retry(state):
  """검증통과 또는 최대 횟수 도달 시 종료"""
  if state['is_verified'] or state['attempt'] >= 3:
    return END
  return "generate"
  
# 질문 -> LLM 답변 생성 -> LLM 답변 검증 -> 검증 통과 
#                                   -> 미통과 -> LLM 답변 생성 -> loop...

graph = StateGraph(VerifyState)
graph.add_node("generate",generate)
graph.add_node("verify",verify)

graph.add_edge(START, "generate")
graph.add_edge("generate", "verify")
graph.add_conditional_edges("verify", should_retry, {"generate":"generate", END:END})

app = graph.compile()
result = app.invoke({
  "question":"파이썬에서 GIL이 무엇이며 멀티쓰레딩에 어떤 영향을 주는지 설명해줘", 
  "answer":"",
  "feedback": "",
  "is_verified":False,
  "attempt":0
})
print("시도횟수",result['attempt'])
print("검증통과",result['is_verified'])
print("최종답변",result['answer'][:300])

app.get_graph().print_ascii()

시도횟수 1
검증통과 True
최종답변 GIL(Global Interpreter Lock)은 파이썬 인터프리터에서 사용되는 뮤텍스(Mutex)입니다. 이는 파이썬 인터프리터가 동시에 하나의 스레드만 실행하도록 제한하는 역할을 합니다. GIL은 파이썬의 메모리 관리가 스레드 안전하지 않기 때문에 도입되었습니다. 이는 파이썬이 멀티쓰레딩 환경에서 예상대로 동작하지 않을 수 있음을 의미합니다.

멀티쓰레딩에 미치는 영향은 다음과 같습니다:

1. **성능 저하**: GIL로 인해 파이썬은 멀티쓰레딩 환경에서 CPU 바운드 작업의 성능을 향상시키기 어렵습니다. 이는 GIL이 동
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| generate |   
+----------+   
      *        
      *        
      *        
  +--------+   
  | verify |   
  +--------+   
      .        
      .        
      .        
 +---------+   
 | __end__ |   
 +---------+   


In [6]:
# 병렬 처리
import time

class AnalysisState(TypedDict):
  text:str
  translated:str 
  summary:str 
  keywords: list[str]

translate_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력:\n{text}")
]) | watson_llm | parser

summarize_chain = ChatPromptTemplate.from_messages([
  ("system", "다음 텍스트를 3문장으로 번역하세요:\n{text}")
]) | watson_llm | parser

keyword_chain = ChatPromptTemplate.from_messages([
  ("system", "키워드 5개 추출:\n{text}")
]) | watson_llm | parser

def translate_node(state):
  print("번역 시작")
  time.sleep(3)
  print("번역 종료")
  return {"translated":translate_chain.invoke({"text":state["text"]})}

def summarize_node(state):
  print("요약 시작")
  time.sleep(3)
  print("요약 종료")
  return {"summary":summarize_chain.invoke({"text":state["translated"]})}

def keyword_node(state):
  print("키워드 시작")
  time.sleep(3)
  print("키워드 종료")
  return {"keywords":keyword_chain.invoke({"text":state["summary"]})}


graph = StateGraph(AnalysisState)
graph.add_node("translate_node", translate_node)
graph.add_node("summarize_node", summarize_node)
graph.add_node("keyword_node", keyword_node)

graph.add_edge(START, "translate_node")
graph.add_edge(START, "summarize_node")
graph.add_edge(START, "keyword_node")
graph.add_edge("translate_node", END)
graph.add_edge( "summarize_node", END)
graph.add_edge( "keyword_node", END)


app = graph.compile()
text = {"text": "Python is a versatile language used in AI and web development"}
result = app.invoke(text)

print("번역", result['translated'][:100])
print("요약", result['summary'][:100])
print("키워드", result['keywords'][:100])

app.get_graph().print_ascii()


NameError: name 'parser' is not defined

- rag LangGraph
  - 사용자 질문 => retrieve => generate => END

In [71]:
# STEP1: 문서로드
loader = PyPDFLoader("./data/Summary of ChatGPTGPT-4 Research.pdf")

# STEP2: 문서분할
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(loader.load())
 
# STEP3: 인덱싱 - 임베딩

# STEP4: 백터스토어(Chroma or FAISS)
vectorstore= Chroma.from_documents(
  documents=chunks, embedding=watson_embedding, persist_directory="./db/chroma_db", collection_name="research"
)

# STEP5: as_retriever(): Vector Store을 Retriever 형태로 반환하여 LangChain에 연결
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={'k': 3})



In [8]:
# RAG

from typing import List
from langchain_core.documents import Document

class RAGState(TypedDict):
  query: str
  retrieved_docs:List[Document]
  answer:str
  
def retrieve(state):
  # 기존의 벡터스토어에 질의
  vectorstore = Chroma(collection_name="research", embedding_function=watson_embedding, persist_directory="./db/chroma_db")
  docs = vectorstore.similarity_search(state['query'], k=3)
  return {"retrieved_docs":docs}

def generate(state):
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
  
  prompt = """\
  다음 컨텍스트를 참고하여 잘문에 대답하세요
  컨텍스트에 없는 내용은 모른다고 답하세요
  
  컨텍스트:
  {context}
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(context=context, query=state['query']))
  return {"answer":response.content}

graph = StateGraph(RAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate",generate)

graph.add_edge(START , "retrieve")
graph.add_edge("retrieve" , "generate")
graph.add_edge("generate" , END)

app = graph.compile()
result = app.invoke({"query":"where can i use ChatGPT?"})
print(result['answer'])

app.get_graph().print_ascii
  

제공된 컨텍스트에 따르면, ChatGPT는 다음과 같은 분야에서 사용될 수 있습니다:

1. 교육 분야: 학생들은 ChatGPT를 사용하여 다양한 학문 분야(예: 물리학, 수학, 화학 등)에서 질문에 대한 답변을 찾고, 비교하며, 검증할 수 있습니다.

2. 코드 생성: ChatGPT는 코드 생성 작업에 사용될 수 있지만, 현재로서는 훈련 데이터가 Python, C++, Java와 같은 프로그래밍 언어에 편중되어 있어 다른 프로그래밍 언어나 코딩 스타일에는 적합하지 않을 수 있습니다.

3. 데이터 시각화, 정보 추출, 데이터 강화, 품질 평가, 멀티모달 데이터 처리 등 다양한 데이터 관련 작업에도 ChatGPT가 적용될 수 있습니다.

컨텍스트에는 ChatGPT가 사용될 수 있는 다른 분야에 대한 언급이 없으므로, 위에 언급된 분야 외에는 ChatGPT의 사용 가능성에 대해 답변드리지 못합니다.


<bound method Graph.print_ascii of Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'retrieve': Node(id='retrieve', name='retrieve', data=retrieve(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'generate': Node(id='generate', name='generate', data=generate(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), '__end__': Node(id='__end__', name='__end__', data=None, metadata=None)}, edges=[Edge(source='__start__', target='retrieve', data=None, conditional=False), Edge(source='retrieve', target='generate', data=None, conditional=False), Edge(source='generate', target='__end__', data=None, conditional=False)])>

In [ ]:
# 기존의 벡터스토어에 질의
vectorstore = Chroma(collection_name="research", embedding_function=watson_embedding, persist_directory="./db/chroma_db")
  

In [13]:
# 기본 RAG + 평가
class RAGState(TypedDict):
  query: str
  retrieved_docs:List[Document]
  answer:str
  is_relevant:bool
  retry_count:int

def retrieve(state):
  docs = vectorstore.similarity_search(state['query'], k=3)
  return {"retrieved_docs":docs}

def generate(state):
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
  
  prompt = """\
  다음 컨텍스트를 참고하여 잘문에 대답하세요
  컨텍스트에 없는 내용은 모른다고 답하세요
  
  컨텍스트:
  {context}
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(context=context, query=state['query']))
  return {"answer":response.content}

# 평가 노드
def evaluate(state):
  """답변이 질문과 관련이 있는지 평가"""
  prompt = ChatPromptTemplate.from_template(
    """
    질문:
    {query}
    
    답변
    {answer}
    
    이 답변이 질문에 적절히 대답하고 있나요? 'yes' 또는 'no'로만 답하세요.
    """
  )
  
  response = watson_llm.invoke(prompt.format(query=state['query'], answer=state['answer']))
  is_relevant = 'yes' in response.content.lower()
  return {"is_relevant": is_relevant, "retry_count":state['retry_count'] + 1}

# route - 멈추는 조건 
def should_retry(state):
  """재검색 여부 결정"""
  if state['is_relevant'] or state['retry_count'] >=2:
    return "done"
  return "retry"

# 그래프
graph = StateGraph(RAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate",evaluate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", should_retry, {"retry": "retrieve", "done":END})

app = graph.compile()
result = app.invoke({"query": "where can i use ChatGPT?", "retry_count":0})

print(result['answer'])
app.get_graph().print_ascii()


제공된 컨텍스트에 따르면, ChatGPT는 다음과 같은 분야에서 사용될 수 있습니다:

1. 교육 분야: 학생들은 ChatGPT를 사용하여 다양한 학문 분야(예: 물리학, 수학, 화학 등)에서 질문에 대한 답변을 찾고, 비교하며, 검증할 수 있습니다.

2. 코드 생성: ChatGPT는 코드 생성 작업에 사용될 수 있지만, 현재로서는 훈련 데이터가 Python, C++, Java와 같은 프로그래밍 언어에 편중되어 있어 다른 프로그래밍 언어나 코딩 스타일에는 적합하지 않을 수 있습니다.

3. 데이터 시각화, 정보 추출, 데이터 강화, 품질 평가, 멀티모달 데이터 처리 등 다양한 데이터 관련 작업에도 ChatGPT가 적용될 수 있습니다.

컨텍스트에는 ChatGPT가 사용될 수 있는 다른 분야에 대한 언급이 없으므로, 위에 언급된 분야 외에는 ChatGPT의 사용 가능성에 대해 답변드리지 못합니다.
           +-----------+       
           | __start__ |       
           +-----------+       
                  *            
                  *            
                  *            
            +----------+       
            | retrieve |       
            +----------+       
           ***        ...      
          *              .     
        **                ...  
+----------+                 . 
| generate |              ...  
+----------+             .     
           ***        ...      
              *      .         
    

#### 검색 품질 개선
- Multi-Query: 여러 관점의 퀄리로 검색(모호한 질문, 넓은 검색 범위 필요)

In [23]:
class MultiQueryState(TypedDict):
  query: str
  sub_queries: List[str]
  retrieve_docs: List[Document]
  answer: str 

def generate_sub_queries(state):
  """원본 질문을 여러 관점의 하위 쿼리로 분해"""
  prompt = """\
    다음 질문에 대해 서로 다른 관점의 검색 쿼리 3개를 생성하세요.
    각 쿼리를 줄바꿈으로 구분하세요.
    
    원본 질문:{query}
    """
    
  response = watson_llm.invoke(prompt.format(query=state['query']))
  #sub_query
  sub_queries = [q for q in response.content.strip().split("\n") if q.strip()]
  print(f"sub queries {sub_queries}")
  
  return {"sub_queries": sub_queries}


def multi_retrieve(state):
  """각 하위 쿼리로 검색하고 결과 합치기"""
  
  all_docs = []
  seen_contents = set() # 중복 제거하고 담아놓기
  
  for sub_query in state['sub_queries']:
    docs = vectorstore.similarity_search(sub_query, k=3)
    for doc in docs:
      if doc.page_content not in seen_contents:
        all_docs.append(doc)
        seen_contents.add(doc.page_content) 
    
  return {"retrieved_docs":all_docs} 

def generate(state):
  context = "\n\n".join(doc.page_content for doc in state["retrieve_docs"])
  
  prompt = """\
  다음 컨텍스트를 참고하여 잘문에 대답하세요
  컨텍스트에 없는 내용은 모른다고 답하세요
  
  컨텍스트:
  {context}
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(context=context, query=state['query']))
  return {"answer":response.content}

# 그래프 생성
graph = StateGraph(MultiQueryState)
graph.add_node("generate_sub_queries", generate_sub_queries)
graph.add_node("multi_retrieve", multi_retrieve)
graph.add_node("generate", generate)

graph.add_edge(START, "generate_sub_queries")
graph.add_edge("generate_sub_queries", "multi_retrieve")
graph.add_edge("multi_retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()
result = app.invoke({"query": "where can i use ChatGPT?", "retrieve_docs":""})

print(result)
app.get_graph().print_ascii()

sub queries ['1. "ChatGPT 사용 가능한 플랫폼 및 서비스"', '2. "ChatGPT를 활용할 수 있는 다양한 분야"', '3. "ChatGPT를 이용할 수 있는 국가 및 지역"']
{'query': 'where can i use ChatGPT?', 'sub_queries': ['1. "ChatGPT 사용 가능한 플랫폼 및 서비스"', '2. "ChatGPT를 활용할 수 있는 다양한 분야"', '3. "ChatGPT를 이용할 수 있는 국가 및 지역"'], 'retrieve_docs': '', 'answer': 'ChatGPT는 다양한 플랫폼에서 사용할 수 있습니다. 웹 브라우저를 통해 OpenAI의 공식 웹사이트에서 직접 접근할 수 있으며, 또한 다양한 애플리케이션과 서비스에 통합되어 사용되고 있습니다. 예를 들어, 개발자들은 ChatGPT API를 사용하여 자신의 애플리케이션에 챗봇 기능을 추가할 수 있습니다. 하지만, ChatGPT의 사용 가능 여부와 방법은 지속적으로 업데이트되고 있으므로, 최신 정보를 확인하기 위해 OpenAI의 공식 웹사이트나 관련 문서를 참고하는 것이 좋습니다.'}
      +-----------+      
      | __start__ |      
      +-----------+      
            *            
            *            
            *            
+----------------------+ 
| generate_sub_queries | 
+----------------------+ 
            *            
            *            
            *            
   +----------------+    
   | multi_retrieve |    
   +----------------+    
            *            
          

In [25]:
# HyDE
# 질문 => 가상의 정답 문서를 먼저 생성, 가상의 문서를 검색어처럼 사용하여 더 관련성 높은 문서 찾음

class HyDEState(TypedDict):
  query: str
  hypothetical_doc: str
  retrieve_docs: List[Document]
  answer: str 


def generate_hypothetical(state):
  """질문에 대한 가상의 답변 문서를 생성합니다."""
  
  prompt = """\
  다음 질문에 대한 답변이 될만한 문서를 작성하세요.
  실제 정확한 답변이 아니어도 됩니다. 관련 용어와 개념을 포함하면 됩니다.
  (maximum context length is 512 tokens.)
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(query=state['query']))
  return {"hypothetical_doc":response.content}


def hyde_retrieve(state):
  """가상문서를 쿼리로 사용하여 검색"""
  docs = vectorstore.similarity_search(state['hypothetical_doc'], k=3)
  return {"retrieved_docs":docs}


def generate(state):
  context = "\n\n".join(doc.page_content for doc in state["retrieve_docs"])
  
  prompt = """\
  다음 컨텍스트를 참고하여 잘문에 대답하세요
  컨텍스트에 없는 내용은 모른다고 답하세요
  
  컨텍스트:
  {context}
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(context=context, query=state['query']))
  return {"answer":response.content}
  

graph = StateGraph(HyDEState)
graph.add_node("generate_hypothetical",generate_hypothetical)
graph.add_node("hyde_retrieve",hyde_retrieve)
graph.add_node("generate",generate)

graph.add_edge(START, "generate_hypothetical")
graph.add_edge("generate_hypothetical", "hyde_retrieve")
graph.add_edge("hyde_retrieve", "generate")
graph.add_edge("generate", END)

app = graph.compile()
result = app.invoke({"query": "where can i use ChatGPT?", "retrieve_docs":""})
print(result["answer"])
app.get_graph().print_ascii()

ChatGPT는 다양한 플랫폼에서 사용할 수 있습니다. 웹 브라우저를 통해 OpenAI의 공식 웹사이트에서 직접 접근할 수 있으며, 또한 다양한 애플리케이션과 서비스에 통합되어 사용되고 있습니다. 예를 들어, 개발자들은 ChatGPT API를 사용하여 자신의 애플리케이션에 챗봇 기능을 추가할 수 있습니다. 하지만, ChatGPT의 사용 가능 여부와 방법은 지속적으로 업데이트되고 있으므로, 최신 정보를 확인하기 위해 OpenAI의 공식 웹사이트나 관련 문서를 참고하는 것이 좋습니다.
      +-----------+        
      | __start__ |        
      +-----------+        
            *              
            *              
            *              
+-----------------------+  
| generate_hypothetical |  
+-----------------------+  
            *              
            *              
            *              
    +---------------+      
    | hyde_retrieve |      
    +---------------+      
            *              
            *              
            *              
      +----------+         
      | generate |         
      +----------+         
            *              
            *              
            *              
       +---------+         
       | __end__ |         


In [27]:
# Self refine rag
# 답변을 생성 -> 평가 -> 재검색 or 답변 수정
# 스스로 평가하고 답변을 바꿔보고

class SelfRefineRAGState(TypedDict):
  query: str
  retrieved_docs:List[Document]
  answer:str
  evaluation:str
  retry_count:int

def retrieve(state):
  docs = vectorstore.similarity_search(state['query'], k=3)
  return {"retrieved_docs":docs}

def generate(state):
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
  
  prompt = """\
  다음 컨텍스트를 참고하여 질문에 대답하세요
  컨텍스트에 없는 내용은 모른다고 답하세요
  컨텍스트에 정보가 부족하면 그 사실을 명시하세요
  
  컨텍스트:
  {context}
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(context=context, query=state['query']))
  return {"answer":response.content}

# 평가 노드
def evaluate(state):
  """생성한 답변이 답변의 충실도와 관련성을 평가"""
  
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
  prompt = ChatPromptTemplate.from_template(
    """
    질문:
    {query}
    
    컨텍스트:
    {context}
    
    답변
    {answer}
    
    반드시 아래 둘 중 하나로만 답하세요
    'sufficient'
    'insufficient'
    """
  )
  
  response = watson_llm.invoke(prompt.format(query=state['query'], answer=state['answer'], context=context))
  content = response.content.lower().strip()
  evaluation = "insufficient" if content.startswith("insufficient") else "sufficient"

  return {"evaluation":evaluation}

def refine(state):
  """평가 결과를 반영하여 답변을 개선"""
  
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])

  prompt = """\
  다음 답변을 개선하세요.
  
  원래 질문:
  {query}

  컨텍스트:
  {context}
  
  이전 답변:
  {answer}
  
  컨텍스트에 더 충실하고 질문에 더 정확히 답하도록 수정하세요
  """

  response = watson_llm.invoke(prompt.format(context=context, query=state['query'], answer=state['answer']))
  return {"answer":response.content, "retry_count":state["retry_count"]+1}

# route 
def route_after_eval(state):
  """평가 후 재검색 여부 결정"""
  if state['evaluation'] == "sufficient" or state['retry_count'] >=2:
    return "done"
  return "refine"

# 그래프
graph = StateGraph(SelfRefineRAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate",evaluate)
graph.add_node("refine",refine)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_after_eval, {"refine": "retrieve", "done":END})
graph.add_edge("refine", "evaluate")

app = graph.compile()
result = app.invoke({"query": "where can i use ChatGPT?", "retry_count":0})

print(result['answer'])
app.get_graph().print_ascii()

Based on the provided context, ChatGPT can be used in the following areas:

1. Education: ChatGPT is commonly used for question and answering testing in the education sector. It can help users learn, compare, and verify answers for different academic subjects such as physics, mathematics, and chemistry.

2. Code generation: ChatGPT has applications in code generation, although there are still some challenges. Its application scope is limited as its training data is biased towards programming languages like Python, C++, and Java, which may make it unsuitable for some programming languages or coding styles.

3. Data visualization, information extraction, data enhancement, quality assessment, and multimodal data processing: ChatGPT shows a wide range of applications in these areas related to data processing and analysis.

The context does not mention any other specific areas where ChatGPT can be used. If you have a particular application in mind that is not covered in the provided context

In [32]:
# 질문으로 재작성 해보기
# Self refine rag
# 답변을 생성 -> 평가 -> 부족 -> 질문 개선 -> 검색  -> 새문서 답변 -> 평가 -> loop


class SelfRAGState(TypedDict):
  query: str
  retrieved_docs:List[Document]
  answer:str
  evaluation:str
  retry_count:int

def retrieve(state):
  docs = vectorstore.similarity_search(state['query'], k=3)
  return {"retrieved_docs":docs}

def generate(state):
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
  
  prompt = """\
  다음 컨텍스트를 참고하여 질문에 대답하세요
  컨텍스트에 없는 내용은 모른다고 답하세요
  컨텍스트에 정보가 부족하면 그 사실을 명시하세요
  
  컨텍스트:
  {context}
  
  질문:
  {query}
  """
  
  response = watson_llm.invoke(prompt.format(context=context, query=state['query']))
  return {"answer":response.content}

# 평가 노드
def evaluate(state):
  """생성한 답변이 답변의 충실도와 관련성을 평가"""
  
  context = "\n\n".join(doc.page_content for doc in state["retrieved_docs"])
  prompt = ChatPromptTemplate.from_template(
    """
    질문:
    {query}
    
    컨텍스트:
    {context}
    
    답변
    {answer}
    
    생성한 답변이 답변의 충실도와 관련성을 평가합니다.
    반드시 아래 둘 중 하나로만 답하세요
    'sufficient'
    'insufficient'
    """
  )
  
  response = watson_llm.invoke(prompt.format(query=state['query'], answer=state['answer'], context=context))
  content = response.content.lower().strip()
  evaluation = "insufficient" if content.startswith("insufficient") else "sufficient"

  return {"evaluation":evaluation}

def rewrite_query(state):
  """평가결과를 반영하여 질문을 개선합니다"""

  prompt = """\
  다음 질문을 개선하세요.
  
  원래 질문:
  {query}
  
  검색 결과가 충분하지 않습니다.
  더 구체적이고 검색하기 좋은 질문으로 재작성하세요.
  질문만 출력하세요.
  """

  response = watson_llm.invoke(prompt.format(query=state['query']))
  return {"query":response.content, "retry_count":state["retry_count"]+1}

# route 
def route_after_eval(state):
  """평가 후 재검색 여부 결정"""
  if state['evaluation'] == "sufficient" or state['retry_count'] >=2:
    return "done"
  return "retry"

# 그래프
graph = StateGraph(SelfRefineRAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate", generate)
graph.add_node("evaluate",evaluate)
graph.add_node("rewrite_query",rewrite_query)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate", route_after_eval, {"done":END, "retry": "retrieve"})
graph.add_edge("rewrite_query", "evaluate")

app = graph.compile()
result = app.invoke({"query": "where can i use ChatGPT?", "retry_count":0})

print(result['answer'])
app.get_graph().print_ascii()


Based on the provided context, ChatGPT can be used in the following areas:

1. Education: ChatGPT is commonly used for question and answering testing in the education sector. It can help users learn, compare, and verify answers for different academic subjects such as physics, mathematics, and chemistry.

2. Code generation: ChatGPT has applications in code generation, although there are still some challenges. Its application scope is limited as its training data is biased towards programming languages like Python, C++, and Java, which may make it unsuitable for some programming languages or coding styles.

3. Data visualization, information extraction, data enhancement, quality assessment, and multimodal data processing: ChatGPT shows a wide range of applications in these areas related to data processing and analysis.

The context does not mention any other specific areas where ChatGPT can be used. If you have a particular application in mind that is not covered in the provided context

#### LLM 애플리케이션 성능 최적화 전략
- 1. 비용절감
  - InMemoryCache, SQLiteCache 모델 겨량화
- 2. 응답속도 향상
  - RunnableParallel, batch, Streaming
- 3. 처리량 향상
  - ainvoke(), asyncio.gather()

In [34]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache
import time

text ="파이썬이란 무엇인가요?"

print("========== 캐시없음 ==========")
set_llm_cache(None)

start = time.time()
r1 = watson_llm.invoke(text)
print(f"1회차: {time.time()-start:.2f}초")

start = time.time()
r1 = watson_llm.invoke(text)
print(f"2회차: {time.time()-start:.2f}초")

# InMemoryCache
print("========== InMemoryCache ==========")
set_llm_cache(InMemoryCache())

start = time.time()
r2 = watson_llm.invoke(text)
print(f"1회차: {time.time()-start:.2f}초")

start = time.time()
r2 = watson_llm.invoke(text)
print(f"2회차: {time.time()-start:.2f}초")

print("========== SQLiteCache ==========")
set_llm_cache(SQLiteCache(database_path="./db/llm_cache.db"))

start = time.time()
r3 = watson_llm.invoke("LangChain이란?")
print(f"1회차: {time.time()-start:.2f}초")

start = time.time()
r3 = watson_llm.invoke("LangChain이란?")
print(f"2회차: {time.time()-start:.2f}초")

========== 캐시없음 ==========
1회차: 5.17초
2회차: 3.52초
========== InMemoryCache ==========
1회차: 3.94초
2회차: 0.00초
========== SQLiteCache ==========
1회차: 2.28초
2회차: 0.00초


/Users/sunahgwak/Documents/Repository/source/ollama/.venv/lib/python3.12/site-packages/langchain_community/cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
